# Classification Quickstart

End-to-end walkthrough of the Phase 1 platform: load a YAML config, inspect the data
pipeline, build a timm-backed model through the registry, fine-tune on CIFAR-10, and
evaluate the best checkpoint.

Setup (from the repo root):
```bash
uv pip install -e ".[dev,notebooks]"
```

In [ ]:
import sys
from pathlib import Path

# Run from the repo root so configs/ and data/ resolve
if Path.cwd().name == "notebooks":
    %cd ..

import torch

print(f"torch {torch.__version__} | cuda: {torch.cuda.is_available()}")

## 1. Load the experiment config

Every experiment is a YAML file parsed into typed dataclasses. Overrides use dotted
keys — here we shrink the run so it finishes in minutes on a laptop (drop the
overrides on a GPU box for the full 10-epoch fine-tune).

In [ ]:
from image_analytics.core.config import load_config

config = load_config(
    "configs/classification/cifar10_resnet18.yaml",
    overrides=[
        "training.epochs=2",
        "data.image_size=96",      # smaller than the full 224 fine-tune resolution
        "data.batch_size=64",
        "data.num_workers=2",
        "experiment_name=cifar10_quickstart",
    ],
)
config.training

## 2. Inspect the data pipeline

`build_dataset` resolves the dataset through the `DATASETS` registry; transforms are
torchvision v2 pipelines built from the config (RandomResizedCrop + flip for training).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from image_analytics.data.datasets import build_dataset
from image_analytics.data.transforms import IMAGENET_MEAN, IMAGENET_STD, build_transforms

train_tf = build_transforms(config.data.image_size, train=True)
train_ds = build_dataset(config.data, split="train", transform=train_tf)
classes = train_ds.classes

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
mean, std = np.array(IMAGENET_MEAN), np.array(IMAGENET_STD)
for ax, idx in zip(axes.flat, range(12)):
    img, label = train_ds[idx]
    img = (img.permute(1, 2, 0).numpy() * std + mean).clip(0, 1)  # un-normalize
    ax.imshow(img)
    ax.set_title(classes[label], fontsize=9)
    ax.axis("off")
plt.tight_layout()

## 3. Build the model through the registry

`build_model` resolves the backbone from the `BACKBONES` registry (or any timm model
name) and attaches a linear head. Swapping architectures is a one-line config change —
try `model.backbone.name=convnext_tiny` or any of timm's 900+ models.

In [ ]:
from image_analytics.classification import build_model
from image_analytics.core.registry import BACKBONES

print(f"Registered backbones: {sorted(BACKBONES)}\n")

model = build_model(config.model)
n_params = sum(p.numel() for p in model.parameters())
print(f"{config.model.backbone.name} -> {config.model.num_classes} classes | {n_params:,} params")

## 4. Train

`run(config)` wires everything: dataloaders, optimizer + cosine schedule with warmup,
evaluator, and callbacks (logging, best-checkpoint saving, optional early stopping).
This is exactly what `python scripts/train.py --config ...` executes.

In [ ]:
from image_analytics.classification.train import run

metrics = run(config)
metrics

## 5. Evaluate the best checkpoint

In [ ]:
from torch.utils.data import DataLoader

from image_analytics.core.evaluator import ClassificationEvaluator
from image_analytics.core.trainer import Trainer

ckpt = Path(config.output_dir) / config.experiment_name / "checkpoints" / "best.pt"

val_tf = build_transforms(config.data.image_size, train=False)
val_ds = build_dataset(config.data, split="val", transform=val_tf)
val_loader = DataLoader(val_ds, batch_size=config.data.batch_size, num_workers=2)

best_model = build_model(config.model)
trainer = Trainer(
    best_model,
    evaluator=ClassificationEvaluator(config.model.num_classes, topk=(1, 5)),
)
trainer.load_checkpoint(ckpt, resume=False)
trainer.validate(val_loader)

## 6. Look at predictions

In [ ]:
images, labels = next(iter(val_loader))
probs = trainer.module.eval().predict(images[:8].to(trainer.device)).cpu()

fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for ax, img, label, p in zip(axes, images, labels, probs):
    pred = int(p.argmax())
    img = (img.permute(1, 2, 0).numpy() * std + mean).clip(0, 1)
    ax.imshow(img)
    color = "green" if pred == int(label) else "red"
    ax.set_title(f"{classes[pred]} ({p[pred]:.0%})", fontsize=9, color=color)
    ax.axis("off")
plt.tight_layout()

## Next steps

- **Swap backbones**: `model.backbone.name=convnext_tiny` (or any timm name)
- **Linear probing**: `model.freeze_backbone()` to train only the head on frozen
  DINOv2 features (`model.backbone.name=dinov2_small`)
- **Multispectral**: see `configs/classification/multispectral_resnet50.yaml` — 13-band
  GeoTIFFs with percentile normalization and per-band channel attention
- **Multi-label**: `model.name=multilabel_classifier` + the `multilabel_csv` dataset
- **Multi-GPU**: `torchrun --nproc_per_node=4 scripts/train.py --config ...`